In [1]:
import os
import math
import re
from   random import *
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import datasets
import numpy as np

# Set GPU device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:

torch.cuda.empty_cache()

### Train, Test, Validation

In [3]:
snli = datasets.load_dataset('snli')
mnli = datasets.load_dataset('glue', 'mnli')
mnli['train'].features, snli['train'].features

({'premise': Value('string'),
  'hypothesis': Value('string'),
  'label': ClassLabel(names=['entailment', 'neutral', 'contradiction']),
  'idx': Value('int32')},
 {'premise': Value('string'),
  'hypothesis': Value('string'),
  'label': ClassLabel(names=['entailment', 'neutral', 'contradiction'])})

In [4]:
# List of datasets to remove 'idx' column from
mnli.column_names.keys()

dict_keys(['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched'])

In [5]:
# Remove 'idx' column from each dataset
for column_names in mnli.column_names.keys():
    mnli[column_names] = mnli[column_names].remove_columns('idx')

In [6]:

mnli.column_names.keys()

dict_keys(['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched'])

In [7]:
np.unique(mnli['train']['label']), np.unique(snli['train']['label'])
#snli also have -1

(array([0, 1, 2]), array([-1,  0,  1,  2]))

In [8]:
# there are -1 values in the label feature, these are where no class could be decided so we remove
snli = snli.filter(
    lambda x: 0 if x['label'] == -1 else 1
)

In [9]:
np.unique(mnli['train']['label']), np.unique(snli['train']['label'])

(array([0, 1, 2]), array([0, 1, 2]))

In [10]:
from datasets import DatasetDict
# Merge the two DatasetDict objects
raw_dataset = DatasetDict({
    'train': datasets.concatenate_datasets([snli['train'], mnli['train']]).shuffle(seed=55).select(list(range(1000))),
    'test': datasets.concatenate_datasets([snli['test'], mnli['test_mismatched']]).shuffle(seed=55).select(list(range(100))),
    'validation': datasets.concatenate_datasets([snli['validation'], mnli['validation_mismatched']]).shuffle(seed=55).select(list(range(1000)))
})
#remove .select(list(range(1000))) in order to use full dataset
# Now, merged_dataset_dict contains the combined datasets from snli and mnli
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 1000
    })
})

#### 2. Preprocessing

In [11]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [12]:
def preprocess_function(examples):
    max_seq_length = 128
    padding = 'max_length'
    # Tokenize the premise
    premise_result = tokenizer(
        examples['premise'], padding=padding, max_length=max_seq_length, truncation=True)
    #num_rows, max_seq_length
    # Tokenize the hypothesis
    hypothesis_result = tokenizer(
        examples['hypothesis'], padding=padding, max_length=max_seq_length, truncation=True)
    #num_rows, max_seq_length
    # Extract labels
    labels = examples["label"]
    #num_rows
    return {
        "premise_input_ids": premise_result["input_ids"],
        "premise_attention_mask": premise_result["attention_mask"],
        "hypothesis_input_ids": hypothesis_result["input_ids"],
        "hypothesis_attention_mask": hypothesis_result["attention_mask"],
        "labels" : labels
    }

tokenized_datasets = raw_dataset.map(
    preprocess_function,
    batched=True,
)

tokenized_datasets = tokenized_datasets.remove_columns(['premise','hypothesis','label'])
tokenized_datasets.set_format("torch")

In [13]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['premise_input_ids', 'premise_attention_mask', 'hypothesis_input_ids', 'hypothesis_attention_mask', 'labels'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['premise_input_ids', 'premise_attention_mask', 'hypothesis_input_ids', 'hypothesis_attention_mask', 'labels'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['premise_input_ids', 'premise_attention_mask', 'hypothesis_input_ids', 'hypothesis_attention_mask', 'labels'],
        num_rows: 1000
    })
})

### Data loader

In [14]:
from torch.utils.data import DataLoader

# initialize the dataloader
batch_size = 8
train_dataloader = DataLoader(
    tokenized_datasets['train'], 
    batch_size=batch_size, 
    shuffle=True
)
eval_dataloader = DataLoader(
    tokenized_datasets['validation'], 
    batch_size=batch_size
)
test_dataloader = DataLoader(
    tokenized_datasets['test'], 
    batch_size=batch_size
)


In [15]:
for batch in train_dataloader:
    print(batch['premise_input_ids'].shape)
    print(batch['premise_attention_mask'].shape)
    print(batch['hypothesis_input_ids'].shape)
    print(batch['hypothesis_attention_mask'].shape)
    print(batch['labels'].shape)
    break

torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8, 128])
torch.Size([8])


### Model

In [16]:
from bert import *
checkpoint = torch.load("model/bert_model.pth", map_location=device)

params = checkpoint["params"]
state  = checkpoint["state_dict"]

model = BERT(
    **params,
    device=device
).to(device)

model.load_state_dict(state)
model.eval()


BERT(
  (embedding): Embedding(
    (tok_embed): Embedding(110759, 768)
    (pos_embed): Embedding(128, 768)
    (seg_embed): Embedding(2, 768)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (layers): ModuleList(
    (0-11): 12 x EncoderLayer(
      (enc_self_attn): MultiHeadAttention(
        (W_Q): Linear(in_features=768, out_features=768, bias=True)
        (W_K): Linear(in_features=768, out_features=768, bias=True)
        (W_V): Linear(in_features=768, out_features=768, bias=True)
      )
      (pos_ffn): PoswiseFeedForwardNet(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (fc2): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
  )
  (fc): Linear(in_features=768, out_features=768, bias=True)
  (activ): Tanh()
  (linear): Linear(in_features=768, out_features=768, bias=True)
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
  (de

Pooling

In [17]:
# define mean pooling function
def mean_pool(token_embeds, attention_mask):
    # reshape attention_mask to cover 768-dimension embeddings
    in_mask = attention_mask.unsqueeze(-1).expand(
        token_embeds.size()
    ).float()
    # perform mean-pooling but exclude padding tokens (specified by in_mask)
    pool = torch.sum(token_embeds * in_mask, 1) / torch.clamp(
        in_mask.sum(1), min=1e-9
    )
    return pool

 Loss Function

In [18]:
def configurations(u,v):
    # build the |u-v| tensor
    uv = torch.sub(u, v)   # batch_size,hidden_dim
    uv_abs = torch.abs(uv) # batch_size,hidden_dim
    
    # concatenate u, v, |u-v|
    x = torch.cat([u, v, uv_abs], dim=-1) # batch_size, 3*hidden_dim
    return x

def cosine_similarity(u, v):
    dot_product = np.dot(u, v)
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    similarity = dot_product / (norm_u * norm_v)
    return similarity

In [19]:
classifier_head = torch.nn.Linear(768*3, 3).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
optimizer_classifier = torch.optim.Adam(classifier_head.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

In [20]:
from transformers import get_linear_schedule_with_warmup

# and setup a warmup for the first ~10% steps
total_steps = int(len(raw_dataset) / batch_size)
warmup_steps = int(0.1 * total_steps)
scheduler = get_linear_schedule_with_warmup(
		optimizer, num_warmup_steps=warmup_steps,
  	num_training_steps=total_steps - warmup_steps
)

# then during the training loop we update the scheduler per step
scheduler.step()

scheduler_classifier = get_linear_schedule_with_warmup(
		optimizer_classifier, num_warmup_steps=warmup_steps,
  	num_training_steps=total_steps - warmup_steps
)

# then during the training loop we update the scheduler per step
scheduler_classifier.step()

C:\Users\User\Miniconda3\envs\myConda\lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Training

In [21]:
max_seq_length = 128

In [22]:
from tqdm.auto import tqdm

num_epoch = 50
# 1 epoch should be enough, increase if wanted
for epoch in range(num_epoch):
    model.train()
    classifier_head.train()
    # initialize the dataloader loop with tqdm (tqdm == progress bar)
    for step, batch in enumerate(tqdm(train_dataloader, leave=True)):
        # zero all gradients on each new step
        optimizer.zero_grad()
        optimizer_classifier.zero_grad()

        # prepare batches and more all to the active device
        inputs_ids_a = batch['premise_input_ids'].to(device)
        inputs_ids_b = batch['hypothesis_input_ids'].to(device)
        attention_a = batch['premise_attention_mask'].to(device)
        attention_b = batch['hypothesis_attention_mask'].to(device)
        segment_ids = torch.zeros(batch_size, max_seq_length, dtype=torch.int32).to(device)  # each input contains only one sentence hence we define them all as sentence '0'
        label = batch['labels'].to(device)

        # extract token embeddings from BERT at last_hidden_state
        u_last_hidden_state = model.get_last_hidden_state(inputs_ids_a, segment_ids)
        v_last_hidden_state = model.get_last_hidden_state(inputs_ids_b, segment_ids)

        # get the mean pooled vectors
        u_mean_pool = mean_pool(u_last_hidden_state, attention_a) # batch_size, hidden_dim
        v_mean_pool = mean_pool(v_last_hidden_state, attention_b) # batch_size, hidden_dim

        # build the |u-v| tensor
        uv = torch.sub(u_mean_pool, v_mean_pool)   # batch_size,hidden_dim
        uv_abs = torch.abs(uv) # batch_size,hidden_dim

        # concatenate u, v, |u-v|
        x = torch.cat([u_mean_pool, v_mean_pool, uv_abs], dim=-1) # batch_size, 3*hidden_dim

        # process concatenated tensor through classifier_head
        x = classifier_head(x) #batch_size, classifer

        # calculate the 'softmax-loss' between predicted and true label
        loss = criterion(x, label)

        # using loss, calculate gradients and then optimizerize
        loss.backward()
        optimizer.step()
        optimizer_classifier.step()

        scheduler.step() # update learning rate scheduler
        scheduler_classifier.step()

    print(f'Epoch: {epoch + 1} | loss = {loss.item():.6f}')

  0%|          | 0/125 [00:00<?, ?it/s]

Epoch: 1 | loss = 2.572585


  0%|          | 0/125 [00:00<?, ?it/s]

Epoch: 2 | loss = 2.161503


  0%|          | 0/125 [00:00<?, ?it/s]

...

Epoch: 48 | loss = 2.371557


  0%|          | 0/125 [00:00<?, ?it/s]

Epoch: 49 | loss = 3.251699


  0%|          | 0/125 [00:00<?, ?it/s]

Epoch: 50 | loss = 3.000608


In [23]:
labels = []
predictions = []
probabilities = []
classes = ["entailment", "neutral", "contradiction"]

In [24]:
model.eval()
classifier_head.eval()
total_similarity = 0
with torch.no_grad():
    for step, batch in enumerate(eval_dataloader):
        # Move batches to the active device
        inputs_ids_a = batch['premise_input_ids'].to(device)
        inputs_ids_b = batch['hypothesis_input_ids'].to(device)
        attention_a = batch['premise_attention_mask'].to(device)
        attention_b = batch['hypothesis_attention_mask'].to(device)
        segment_ids = torch.zeros(inputs_ids_a.shape[0], inputs_ids_a.shape[1], dtype=torch.int32).to(device)
        label = batch['labels'].to(device)

        # Extract token embeddings from BERT
        u = model.get_last_hidden_state(inputs_ids_a, segment_ids)  # (batch_size, seq_len, hidden_dim)
        v = model.get_last_hidden_state(inputs_ids_b, segment_ids)  # (batch_size, seq_len, hidden_dim)

        # Get the mean pooled vectors (Keep them as Tensors)
        u_mean_pool = mean_pool(u, attention_a)  # (batch_size, hidden_dim)
        v_mean_pool = mean_pool(v, attention_b)  # (batch_size, hidden_dim)

        # Computing cosine similarity
        similarity_score = cosine_similarity(u_mean_pool.cpu().numpy().reshape(-1), v_mean_pool.cpu().numpy().reshape(-1))
        total_similarity += similarity_score

        # Concatenate [u, v, |u - v|]
        uv_abs = torch.abs(u_mean_pool - v_mean_pool)  # [batch_size, hidden_dim]
        x = torch.cat([u_mean_pool, v_mean_pool, uv_abs], dim=-1)  # [batch_size, 3*hidden_dim]

        # Classification
        logit_fn = classifier_head(x)  # (batch_size, num_classes)
        probs = torch.nn.functional.softmax(logit_fn, dim=-1)

        preds = torch.argmax(logit_fn, dim=-1)

        labels.extend(label.cpu().tolist())
        probabilities.extend(probs.cpu().tolist())
        predictions.extend(preds.cpu().tolist())

average_similarity = total_similarity / len(eval_dataloader)
print(f"Average Cosine Similarity: {average_similarity:.4f}")

Average Cosine Similarity: 0.9969


In [25]:
from sklearn.metrics import classification_report

print(classification_report(labels, predictions, target_names=classes))

               precision    recall  f1-score   support

   entailment       0.33      0.07      0.12       338
      neutral       0.33      0.92      0.48       328
contradiction       0.33      0.01      0.02       334

     accuracy                           0.33      1000
    macro avg       0.33      0.33      0.21      1000
 weighted avg       0.33      0.33      0.20      1000



In [26]:
# saving the model
torch.save([model.params, model.state_dict()], 'model/sen_bert.pth')

### Inference

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_similarity(model, tokenizer, sentence_a, sentence_b, device):
    # Tokenize
    inputs_a = tokenizer(
        sentence_a,
        return_tensors='pt',
        max_length=max_seq_length,
        truncation=True,
        padding='max_length'
    ).to(device)

    inputs_b = tokenizer(
        sentence_b,
        return_tensors='pt',
        max_length=max_seq_length,
        truncation=True,
        padding='max_length'
    ).to(device)

    # Extract inputs
    input_ids_a = inputs_a['input_ids']
    attention_a = inputs_a['attention_mask']
    input_ids_b = inputs_b['input_ids']
    attention_b = inputs_b['attention_mask']

    segment_ids = torch.zeros(
        1, max_seq_length, dtype=torch.int32
    ).to(device)

    # Forward pass
    u = model.get_last_hidden_state(input_ids_a, segment_ids)
    v = model.get_last_hidden_state(input_ids_b, segment_ids)

    # Mean pooling → (hidden_dim,)
    u = mean_pool(u, attention_a).detach().cpu().numpy()
    v = mean_pool(v, attention_b).detach().cpu().numpy()

    # 🔥 FIX: reshape to 2D
    similarity_score = cosine_similarity(
        u.reshape(1, -1),
        v.reshape(1, -1)
    )[0][0]

    return similarity_score


In [28]:
# Example usage:
sentence_a = 'Your contribution helped make it possible for us to provide our students with a quality education.'
sentence_b = "Your contributions were of no help with our students' education."
similarity = calculate_similarity(model, tokenizer, sentence_a, sentence_b, device)
print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.9994


### Task-3

In [29]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')
pre_trained_model = AutoModel.from_pretrained('sentence-transformers/all-mpnet-base-v2')

In [30]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

In [31]:
pos_sentence = ["The cat is sleeping on the couch.", "The feline is resting on the sofa."]
opp_sentence = ["He is very punctual and reliable.", "You can never count on him to be on time."]

In [32]:
encoded_input = tokenizer(pos_sentence, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    model_output = pre_trained_model(**encoded_input)

In [33]:
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

sent_a_emb = sentence_embeddings[0].cpu().numpy().reshape(1, -1)
sent_b_emb = sentence_embeddings[1].cpu().numpy().reshape(1, -1)
cosine_similarity(sent_a_emb, sent_b_emb)[0][0]

np.float32(0.73199296)

In [34]:
encoded_input = tokenizer(opp_sentence, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    model_output = pre_trained_model(**encoded_input)

In [35]:
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

sent_a_emb = sentence_embeddings[0].cpu().numpy().reshape(1, -1)
sent_b_emb = sentence_embeddings[1].cpu().numpy().reshape(1, -1)
cosine_similarity(sent_a_emb, sent_b_emb)[0][0]

np.float32(0.48303473)

### Classification Report

| Class          | Precision | Recall | F1-score | Support |
|----------------|-----------|--------|----------|---------|
| entailment     | 0.33      | 0.07   | 0.12     | 338     |
| neutral        | 0.33      | 0.92   | 0.48     | 328     |
| contradiction  | 0.33      | 0.01   | 0.02     | 334     |
|                |           |        |          |         |
| **Accuracy**   |           |        | **0.33** | 1000    |
| **Macro Avg**  | 0.33      | 0.33   | 0.21     | 1000    |
| **Weighted Avg** | 0.33    | 0.33   | 0.20     | 1000    |


## Comparison of our model with pre-trained model

| Model Type | Cosine Similarity (Similar sentence) | Cosine Similarity (Dissisimilar sentence) |
|----------|----------|----------|
| Our Model    | 0.9992    | 0.999    |
| Pre-trained    | 0.731     | 0.483     |

### Observation

In this assignment, the first task involved implementing BERT from scratch, following the reference notebook provided by the professor (BERT-update.ipynb
). The Wikipedia dataset from Hugging Face was used as the training corpus. However, due to hardware and memory constraints, the dataset had to be significantly reduced to 100,000 samples. While this subset enabled training on limited resources, it also restricted the model’s ability to learn rich contextual representations.

During model training, additional challenges were encountered. Because of GPU/CPU memory limitations, the batch size was reduced to 3, and the number of epochs was limited to 7. Initial experiments with a much larger number of epochs (up to 1000) showed that the training loss stopped improving after around 900 epochs and eventually resulted in an out-of-memory error. Consequently, the final training configuration was constrained by computational feasibility rather than optimal learning conditions. As expected, the resulting model demonstrated weak performance during inference, indicating insufficient convergence and representation learning.

In Task 2, a sentence-pair classification model was trained using the SNLI and MNLI datasets, following the reference implementation provided in the professor’s notebook (S-BERT.ipynb
). The objective was to learn semantic relationships between sentence pairs (entailment, neutral, and contradiction). Similar to Task 1, hardware limitations played a significant role. The original batch size of 32 was not feasible on the available device, requiring it to be reduced to 8, which negatively affected training stability and gradient estimation.

In Task 3 (Evaluation and Analysis), the trained custom model exhibited poor classification performance, particularly for entailment and contradiction classes. To contextualize this result, the performance was compared with a pre-trained sentence embedding model (all-mpnet-base-v2) from Hugging Face. The contrast clearly demonstrated the limitations of the custom-trained model and highlighted the importance of large-scale pre-training and sufficient computational resources.

Overall, the main challenges faced in this assignment can be summarized as follows:

- Limited model performance due to under-training
- Small effective dataset size
- Severe hardware and memory constraints